This notebook contains code on collecting public transport data from OSM for the generated grids and then calculating the number of public transport options for each 100m x 100m grid

In [1]:
import geopandas as gpd
import os
import glob
import logging
import osmium
import shapely.geometry
from shapely.geometry import box
import subprocess
import gc
from multiprocess import Pool
import sys

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]

In [2]:
# set path for grid files
# you might have grids generated here from the 'generate_grids.ipynb'
grid_path = 'data/*.parquet'

# create a list of all parquet grid files from the specified directory
grids_list = [parquet for parquet in glob.glob(grid_path)]
print(grids_list)

# configure logging (recommended if you monitor processing over a lot of files)
log_path = 'logs/public_transport.log'

# ensure log directory exists
log_dir = os.path.dirname(log_path)
if not os.path.exists(log_dir):
    os.makedirs(log_dir)
    
logging.basicConfig(filename=log_path, level=logging.INFO,
                    format='%(asctime)s:%(levelname)s:%(message)s', force=True)

['data/grids_100_745.parquet', 'data/grids_100_648.parquet', 'data/grids_100_810.parquet', 'data/grids_100_813.parquet', 'data/grids_100_744.parquet', 'data/grids_100_741.parquet', 'data/grids_100_743.parquet', 'data/grids_100_684.parquet', 'data/grids_100_765.parquet', 'data/grids_100_713.parquet', 'data/grids_100_812.parquet', 'data/grids_100_646.parquet', 'data/grids_100_719.parquet', 'data/grids_100_787.parquet', 'data/grids_100_569.parquet', 'data/grids_100_530.parquet', 'data/grids_100_869.parquet', 'data/grids_100_718.parquet', 'data/grids_100_685.parquet', 'data/grids_100_868.parquet', 'data/grids_100_607.parquet', 'data/grids_100_834.parquet', 'data/grids_100_832.parquet', 'data/grids_100_717.parquet', 'data/grids_100_529.parquet', 'data/grids_100_790.parquet', 'data/grids_100_811.parquet', 'data/grids_100_789.parquet', 'data/grids_100_604.parquet', 'data/grids_100_647.parquet', 'data/grids_100_830.parquet', 'data/grids_100_567.parquet', 'data/grids_100_742.parquet', 'data/gri

In [3]:
def is_public_transport_node(tags):
    """
    Determine if a node has public transport-related tags.
    """
    transport_tags = {
        ("highway", "bus_stop"),
        ("public_transport", "station"),
        ("public_transport", "platform"),
        ("railway", "station"),
        ("railway", "tram_stop"),
        ("railway", "halt"),
        ("railway", "subway_entrance"),
        ("amenity", "bus_station"),
    }
    return any(tags.get(k) == v for k, v in transport_tags)

def extract_transport_nodes_from_pbf(pbf_path):
    """
    Extract public transportation nodes from an OSM PBF file.

    Args:
    - pbf_path (str): Path to the .osm.pbf file.

    Returns:
    - GeoDataFrame of transport nodes.
    """
    transport_nodes = []

    class Handler(osmium.SimpleHandler):
        def node(self, n):
            if n.location.valid() and is_public_transport_node(n.tags):
                transport_nodes.append({
                    "id": n.id,
                    "name": n.tags.get("name"),
                    "geometry": shapely.geometry.Point(n.location.lon, n.location.lat),
                    "type": next((f"{k}={v}" for k, v in n.tags if (k, v) in {
                        ("highway", "bus_stop"),
                        ("public_transport", "station"),
                        ("public_transport", "platform"),
                        ("railway", "station"),
                        ("railway", "tram_stop"),
                        ("railway", "halt"),
                        ("railway", "subway_entrance"),
                        ("amenity", "bus_station"),
                    }), None)
                })

    handler = Handler()
    handler.apply_file(pbf_path, locations=True)

    if not transport_nodes:
        return gpd.GeoDataFrame(columns=["id", "name", "type", "geometry"], geometry="geometry", crs="EPSG:4326")

    return gpd.GeoDataFrame(transport_nodes, crs="EPSG:4326")

In [4]:
def process_grid(grid_path):
    """
    Processes each grid file to query and save public transport data.
    
    Args:
    - grid_path (str): Path to the grid file.
    """
    try:
        grid_number = grid_path.split('_')[-1].split('.')[0]
        output_dir = os.path.join('/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/', 'public_transport_data')
        output_file = f'public_transport_points_{grid_number}.parquet'
        os.makedirs(output_dir, exist_ok=True)

        if os.path.exists(os.path.join(output_dir, output_file)):
            # logging.info(f'Skipping {output_file} as it already exists')
            print(f'Skipping {output_file} as it already exists')
            return

        grid_gdf = gpd.read_parquet(grid_path)
        if 'index' not in grid_gdf.columns:
            grid_gdf.reset_index(inplace=True)

        grid_gdf_4326 = grid_gdf.to_crs('epsg:4326')
        bounds = grid_gdf_4326.total_bounds
        buffered_box = box(*bounds).buffer(0.01)
        # logging.info(f'Started processing grid {grid_number}')
        print(f'Started processing grid {grid_number}')

        # clip .pbf file
        clipped_pbf_path = f'/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/eurostat_grid/grids_100/pbf_extracts/{grid_number}.pbf'
        os.makedirs(os.path.dirname(clipped_pbf_path), exist_ok=True)

        bbox_str = ','.join(map(str, buffered_box.bounds))
        subprocess.run([
            'osmium', 'extract',
            '-b', bbox_str,
            '/rds/user/trpb2/hpc-work/walkability/walkability-obct/data/united-kingdom-261028.osm.pbf',
            '-o', clipped_pbf_path,
            '--overwrite'
        ], check=True)
        print(f'clipped PBF for {grid_number}')

        # process with pyosmium
        data_gdf = extract_transport_nodes_from_pbf(clipped_pbf_path)
        
        if data_gdf.empty:
            # logging.info(f'No public transport data found for grid {grid_path}')
            print(f'No public transport data found for grid {grid_path}')
            return

        data_gdf.to_crs(grid_gdf.crs, inplace=True)
        data_gdf.to_parquet(os.path.join(output_dir, output_file))
        # logging.info(f'Saved public transport data to {output_file}')
        print(f'Saved public transport data to {output_file}')

        joined = gpd.sjoin(grid_gdf, data_gdf, how="left", predicate='intersects')
        node_counts_per_grid = joined.groupby('index')['index_right'].nunique().reset_index(name='pub_trans_count')
        grid_gdf_f = grid_gdf.merge(node_counts_per_grid, on='index', how='left').fillna({'pub_trans_count': 0})
        grid_gdf_f.to_parquet(grid_path)
        # logging.info(f'Successfully processed grid {grid_path}')
        print(f'Successfully processed grid {grid_path}')

        # clean up temp PBF file
        if os.path.exists(clipped_pbf_path):
            os.remove(clipped_pbf_path)
            
        del data_gdf, grid_gdf, grid_gdf_f
        gc.collect()
    except Exception as e:
        # logging.error(f'Error processing grid {grid_path}: {e}')
        print(f'Error processing grid {grid_path}: {e}')

In [5]:
# # sequential
# for elem in grids_list:
#     process_grid(elem)

# parallel processsing if you want to process a lot of files
# need to reduce to 2
num_processes = 2

with Pool(processes=num_processes) as pool:
    pool.map(process_grid, grids_list)

Skipping public_transport_points_765.parquet as it already existsSkipping public_transport_points_745.parquet as it already exists

Skipping public_transport_points_648.parquet as it already exists
Skipping public_transport_points_713.parquet as it already exists
Skipping public_transport_points_810.parquet as it already exists
Skipping public_transport_points_813.parquet as it already exists
Skipping public_transport_points_744.parquet as it already exists
Skipping public_transport_points_812.parquet as it already exists
Skipping public_transport_points_646.parquet as it already exists
Skipping public_transport_points_719.parquet as it already exists
Skipping public_transport_points_787.parquet as it already exists
Skipping public_transport_points_569.parquet as it already exists
Skipping public_transport_points_530.parquet as it already exists
Skipping public_transport_points_869.parquet as it already exists
Skipping public_transport_points_741.parquet as it already exists
Skipping p

In [14]:
import geopandas as gpd

gdf = gpd.read_parquet("data/grids_100_644.parquet")
gdf.head()

,index,geometry,grid_100000_id,slope,dou,ent_5,ent_10,ent_15,ndvi,population,green_area,pub_trans_count
0,0,"POLYGON ((3300000 3300000, 3300100 3300000, 33...",644,3.376099,12,0.374755,0.238191,0.135091,0.601734,0.000000,1653.960423,0
1,1,"POLYGON ((3300000 3300100, 3300100 3300100, 33...",644,3.012833,12,0.448337,0.236169,0.134511,0.620374,0.000000,3692.903849,0
2,2,"POLYGON ((3300000 3300200, 3300100 3300200, 33...",644,3.039882,12,0.520346,0.234974,0.134051,0.711065,0.074744,0.000000,0
3,3,"POLYGON ((3300000 3300300, 3300100 3300300, 33...",644,3.763307,12,0.530854,0.233792,0.133481,0.692741,1.351793,0.000000,0
4,4,"POLYGON ((3300000 3300400, 3300100 3300400, 33...",644,4.854097,12,0.530854,0.232623,0.132916,0.551307,10.757924,0.000000,0
